In [1]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Using backend: {backend.name}")

# 1. Build the simplest possible circuit: put a qubit in superposition and measure it
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

# 2. Transpile it for this specific backend's native gate set
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# 3. Submit the job
sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=100)
print(f"Job ID: {job.job_id()}")
print("Status:", job.status())

Using backend: ibm_marrakesh
Job ID: dah37b7i3e6s738n7tp0
Status: QUEUED


In [2]:
result = job.result()
counts = result[0].data.c.get_counts()
print(counts)

{'0': 45, '1': 55}


In [4]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

In [19]:
import pennylane as qml
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Prepare your QUBO-selected features
quantum_features = ['mean perimeter', 'mean symmetry', 'concavity error', 'worst area']
feature_indices = [list(feature_names).index(f) for f in quantum_features]
X_quantum = X[:, feature_indices]

# 2. Scale features to [0, pi] — angle encoding needs bounded inputs
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_quantum)

# 3. Define the feature map: angle encoding + entanglement (ZZFeatureMap-style)
n_qubits = len(quantum_features)  # 4 qubits, one per feature
dev = qml.device("default.qubit", wires=n_qubits)

def feature_map(x):
    # Layer 1: encode each feature as a rotation on its own qubit
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
        qml.RZ(2 * x[i], wires=i)
    # Layer 2: entangle adjacent qubits, encoding pairwise feature products
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])
        qml.RZ(2 * (np.pi - x[i]) * (np.pi - x[i + 1]), wires=i + 1)
        qml.CNOT(wires=[i, i + 1])

In [6]:
@qml.qnode(dev)
def kernel_circuit(x1, x2):
    # Encode the first data point
    feature_map(x1)
    # Apply the inverse feature map for the second data point
    qml.adjoint(feature_map)(x2)
    # Measure the probability of returning to the all-zeros state
    return qml.probs(wires=range(n_qubits))

def quantum_kernel(x1, x2):
    # The |0...0> probability IS the kernel value (state overlap squared)
    probs = kernel_circuit(x1, x2)
    return probs[0]

In [7]:
print(quantum_kernel(X_scaled[0], X_scaled[1]))
print(quantum_kernel(X_scaled[0], X_scaled[0]))  

0.07643355057720785
0.9999999999999987


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# 2. Build kernel matrices (this computes quantum_kernel for every pair — can take a minute)
def compute_kernel_matrix(A, B):
    return np.array([[quantum_kernel(a, b) for b in B] for a in A])

K_train = compute_kernel_matrix(X_train, X_train)
K_test = compute_kernel_matrix(X_test, X_train)

# 3. Train an SVM using the precomputed quantum kernel
qsvm = SVC(kernel='precomputed')
qsvm.fit(K_train, y_train)

# 4. Evaluate
preds = qsvm.predict(K_test)
acc_qsvm = accuracy_score(y_test, preds)
print(f"Quantum Kernel SVM accuracy: {acc_qsvm:.4f}")

Quantum Kernel SVM accuracy: 0.9006


In [9]:
# Classical SVM (RBF kernel) on the same 4 features, for a fair comparison
classical_svm = SVC(kernel='rbf')
classical_svm.fit(X_train, y_train)
classical_preds = classical_svm.predict(X_test)
acc_classical_svm = accuracy_score(y_test, classical_preds)
print(f"Classical RBF SVM accuracy: {acc_classical_svm:.4f}")

Classical RBF SVM accuracy: 0.9474


In [10]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Using: {backend.name}")

# Real-hardware quantum device via PennyLane's qiskit.remote
# (runs in job mode by default — compatible with the free Open Plan, no Session needed)
dev_real = qml.device('qiskit.remote', wires=n_qubits, backend=backend, shots=200)

@qml.qnode(dev_real)
def kernel_circuit_real(x1, x2):
    feature_map(x1)
    qml.adjoint(feature_map)(x2)
    return qml.probs(wires=range(n_qubits))

def quantum_kernel_real(x1, x2):
    probs = kernel_circuit_real(x1, x2)
    return probs[0]

Using: ibm_fez


In [11]:
import random
random.seed(42)
sample_pairs = [(random.choice(range(len(X_test))), random.choice(range(len(X_train)))) for _ in range(5)]

print(f"{'Pair':<10}{'Simulator':<12}{'Real HW':<12}{'Diff'}")
for i, j in sample_pairs:
    sim_val = quantum_kernel(X_test[i], X_train[j])
    real_val = quantum_kernel_real(X_test[i], X_train[j])
    print(f"({i},{j})    {sim_val:.4f}      {real_val:.4f}      {abs(sim_val - real_val):.4f}")

Pair      Simulator   Real HW     Diff
(163,57)    0.0005      0.0050      0.0045
(6,379)    0.0115      0.0000      0.0115
(70,125)    0.0008      0.0050      0.0042
(57,71)    0.0046      0.0100      0.0054
(26,346)    0.3536      0.3200      0.0336


In [17]:
from qiskit import QuantumCircuit

# --- setup: define backend, transpiler and mitigated sampler used below ---
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
sampler_mitigated = Sampler(mode=backend)
sampler_mitigated.options.dynamical_decoupling.enable = True

def build_kernel_circuit_qiskit(x1, x2):
    qc = QuantumCircuit(n_qubits, n_qubits)

    # --- Forward feature map for x1 ---
    for i in range(n_qubits):
        qc.h(i)
        qc.rz(2 * x1[i], i)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)
        qc.rz(2 * (np.pi - x1[i]) * (np.pi - x1[i + 1]), i + 1)
        qc.cx(i, i + 1)

    # --- Inverse feature map for x2 (reverse order, negated angles) ---
    for i in reversed(range(n_qubits - 1)):
        qc.cx(i, i + 1)
        qc.rz(-2 * (np.pi - x2[i]) * (np.pi - x2[i + 1]), i + 1)
        qc.cx(i, i + 1)
    for i in reversed(range(n_qubits)):
        qc.rz(-2 * x2[i], i)
        qc.h(i)

    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def quantum_kernel_mitigated_manual(x1, x2):
    qc = build_kernel_circuit_qiskit(x1, x2)
    isa_circuit = pm.run(qc)
    job = sampler_mitigated.run([isa_circuit], shots=200)
    result = job.result()[0]
    counts = result.data.c.get_counts()
    total = sum(counts.values())
    zero_state = '0' * n_qubits
    return counts.get(zero_state, 0) / total

print(f"{'Pair':<10}{'Simulator':<12}{'Real HW':<12}{'Real HW+EM':<12}")
for i, j in sample_pairs:
    sim_val = quantum_kernel(X_test[i], X_train[j])
    real_val = quantum_kernel_real(X_test[i], X_train[j])
    mitigated_val = quantum_kernel_mitigated_manual(X_test[i], X_train[j])
    print(f"({i},{j})    {sim_val:.4f}      {real_val:.4f}      {mitigated_val:.4f}")

Pair      Simulator   Real HW     Real HW+EM  
(163,57)    0.0005      0.0100      0.0050
(6,379)    0.0115      0.0000      0.0150
(70,125)    0.0008      0.0000      0.0000
(57,71)    0.0046      0.0150      0.0250
(26,346)    0.3536      0.3550      0.2700


In [21]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = malignant, 1 = benign

df.head(10)  # shows the first 10 patients, all 30 features + the answer

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0
5,12.45,15.70,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,0.07613,...,23.75,103.40,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.12440,0
6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,0.05742,...,27.66,153.20,1606.0,0.1442,0.2576,0.3784,0.1932,0.3063,0.08368,0
7,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,0.07451,...,28.14,110.60,897.0,0.1654,0.3682,0.2678,0.1556,0.3196,0.11510,0
8,13.00,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,0.07389,...,30.73,106.20,739.3,0.1703,0.5401,0.5390,0.2060,0.4378,0.10720,0
9,12.46,24.04,83.97,475.9,0.11860,0.23960,0.22730,0.08543,0.2030,0.08243,...,40.68,97.65,711.4,0.1853,1.0580,1.1050,0.2210,0.4366,0.20750,0
